# Libs

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import warnings
from collections import Counter
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from scipy.stats import kurtosis

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
    balanced_accuracy_score, roc_curve, roc_auc_score
)
from sklearn.metrics import roc_curve, auc as calc_auc, precision_recall_curve, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import metrics, layers, models
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Conv1D, LSTM, GlobalMaxPooling1D,
    LayerNormalization, MultiHeadAttention, Add, Flatten, Concatenate, MaxPooling1D, BatchNormalization, GlobalAveragePooling1D, AveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback
from tensorflow.keras.optimizers import AdamW

print("Versão do TensorFlow:", tf.__version__)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sktime.transformations.panel.rocket import Rocket
from sktime.datatypes._panel._convert import from_long_to_nested

from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute

from futurai_ppd import drop_transitorio_desligado

warnings.filterwarnings('ignore')

# Import dataset

In [ ]:
base_name = 'Depurador 762-28-006 - Cozimento'
timestamp = "Timestamp"

df_dataset = pd.read_csv('data/' + base_name + '.csv', sep=";", decimal=".", encoding="utf-8-sig")
df_dataset[timestamp] = pd.to_datetime(df_dataset[timestamp], format="%Y-%m-%d %H:%M:%S")

## Drop columns with NaN values, constant values or irrelevant to the analysis
df_dataset.drop(columns=["762H0336.PV", "762H0342.PV", "762N0015.SP", "762P0013.SP", "762-34-073.CR", "762N0015.OP", "762F0014.SP"], inplace=True, errors='ignore')

df_dataset.dropna(inplace=True)

print(f"Dataset shape: {df_dataset.shape}")

list_variables = df_dataset.columns.tolist()
df_dataset.head()

## Remove periods Off

In [ ]:
pre_process = []
pp_var_ref_desligado = "762-28-006.CR"
pp_valor_ref_desligado = 5
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 30
pp_pos_corte_transitorio = 30
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

for pro in pre_process:
    df_dataset,_,_ = drop_transitorio_desligado(df_dataset,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
print(f"Dataset shape: {df_dataset.shape}")
df_dataset.head()

## Create label for anomaly

In [ ]:
periodos_de_Falhas = [
    (pd.Timestamp('2024-05-03 11:00:00'), pd.Timestamp('2024-05-03 11:35:00')),
    (pd.Timestamp('2024-06-25 17:20:00'), pd.Timestamp('2024-08-02 14:00:00')),
    (pd.Timestamp('2024-10-19 10:40:00'), pd.Timestamp('2024-10-19 10:50:00')),
    (pd.Timestamp('2024-10-19 10:40:00'), pd.Timestamp('2024-10-19 10:50:00')),
    (pd.Timestamp('2024-10-21 12:00:00'), pd.Timestamp('2024-10-22 00:35:00')),
    (pd.Timestamp('2024-10-24 03:10:00'), pd.Timestamp('2024-10-27 00:00:00')),
    (pd.Timestamp('2024-11-14 06:40:00'), pd.Timestamp('2024-11-14 19:45:00')),
    (pd.Timestamp('2024-11-25 21:45:00'), pd.Timestamp('2024-11-25 22:03:00')),
    (pd.Timestamp('2024-11-27 15:00:00'), pd.Timestamp('2024-11-27 15:07:00')),
    (pd.Timestamp('2024-11-27 16:04:00'), pd.Timestamp('2024-11-27 16:11:00')),
    (pd.Timestamp('2024-11-30 13:30:00'), pd.Timestamp('2024-11-30 15:52:00')),
    (pd.Timestamp('2024-12-09 20:30:00'), pd.Timestamp('2024-12-11 07:00:00')),
    (pd.Timestamp('2024-12-12 20:45:00'), pd.Timestamp('2024-12-12 21:15:00')),
    (pd.Timestamp('2025-03-05 15:17:00'), pd.Timestamp('2025-03-05 15:24:00')),
    (pd.Timestamp('2025-03-15 18:30:00'), pd.Timestamp('2025-03-15 19:15:00')),
    (pd.Timestamp('2025-03-18 11:40:00'), pd.Timestamp('2025-03-18 20:00:00')),
    (pd.Timestamp('2025-06-11 17:25:00'), pd.Timestamp('2025-06-11 17:50:00')),
    (pd.Timestamp('2025-06-16 15:20:00'), pd.Timestamp('2025-06-17 07:38:00')),
]

df_dataset['Falhas'] = 0
for inicio, fim in periodos_de_Falhas:
    df_dataset.loc[(df_dataset[timestamp] >= inicio) & (df_dataset[timestamp] <= fim), 'Falhas'] = 1
df_dataset.head()

## Import TAGs and descriptions

In [ ]:
df_subsistema = pd.read_csv('data/'+ base_name + '_subsistema.csv', sep=";", decimal=".", encoding="utf-8-sig")
df_subsistema

## Plot variables

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['Timestamp'],
    y=df_dataset["762P0034.PV"],
    mode='lines',
    name='762P0034.PV',
    line=dict(color='black')
))

fig.add_trace(go.Scatter(
    x=df_dataset['Timestamp'],
    y=df_dataset["Falhas"],
    mode='lines',
    name='Falhas',
    line=dict(color='red')
))

fig.update_layout(
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

# TSC with DeepLearning Algorithms

## Train/Test Split

#### Train

In [ ]:
start_date = pd.to_datetime("2024-01-01 00:00:00")
end_date = pd.to_datetime("2025-03-12 09:25:00")

mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_train = df_dataset.loc[mask]

df_train.sort_values(by=[timestamp],inplace=True)
print(df_train.shape)
print("Falhas Train:",df_train[df_train["Falhas"]==1].shape[0])
print("Normal Train:",df_train[df_train["Falhas"]==0].shape[0])

#### Test

In [ ]:
start_date = pd.to_datetime("2025-03-14 07:54:00")
end_date = pd.to_datetime("2025-10-30 15:00:00")
mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

df_test.sort_values(by=[timestamp],inplace=True)
print(df_test.shape)
print("Falhas Test:",df_test[df_test["Falhas"]==1].shape[0])
print("Normal Test:",df_test[df_test["Falhas"]==0].shape[0])

### Data Scaling

In [ ]:
# --- Preparação dos dados ---
X_train = df_train.drop(columns=["Timestamp", "Falhas"], axis=1)
y_train = df_train["Falhas"]

X_test = df_test.drop(columns=["Timestamp", "Falhas"], axis=1)
y_test = df_test["Falhas"]

# --- Normalização apenas das features ---
scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

### Reshape data for DL Algs

In [ ]:
# --- Função para criar janelas com timestamp central ---
def create_windows(df, window_size=60, step=30, timestamp_col="Timestamp", label_col="Falhas"):
    """
    Cria janelas deslizantes de dados para séries temporais multivariadas.
    Retorna também o timestamp central de cada janela para posterior visualização.

    Args:
        df (pd.DataFrame): DataFrame contendo as colunas de features, timestamp e label.
        window_size (int): Tamanho da janela temporal.
        step (int): Passo entre as janelas.
        timestamp_col (str): Nome da coluna de timestamp.
        label_col (str): Nome da coluna de rótulo.

    Returns:
        X (np.ndarray): Janelas com shape (n_janelas, window_size, n_features)
        y (np.ndarray): Rótulo principal de cada janela
        timestamps (np.ndarray): Timestamp central de cada janela
    """
    X, y, timestamps = [], [], []
    timestamps_series = df[timestamp_col].reset_index(drop=True)

    for i in range(0, len(df) - window_size + 1, step):
        window = df.iloc[i:i + window_size]

        # Adiciona features e label
        X.append(window.drop([timestamp_col, label_col], axis=1).values)
        y.append(window[label_col].mode()[0])

        # Timestamp central da janela
        central_idx = i + window_size // 2
        timestamps.append(timestamps_series.iloc[central_idx])

    return np.array(X), np.array(y), np.array(timestamps)

# --- Parâmetros das janelas ---
window_size = 60
step = 30

# --- Reconstrução dos DataFrames completos (mantendo o Timestamp) ---
df_train_full = pd.concat([df_train["Timestamp"], X_train, y_train], axis=1)
df_test_full = pd.concat([df_test["Timestamp"], X_test, y_test], axis=1)

# --- Criação das janelas ---
X_window_train, y_window_train, ts_window_train = create_windows(df_train_full, window_size, step)
X_window_test, y_window_test, ts_window_test = create_windows(df_test_full, window_size, step)

print("X_window_train shape:", X_window_train.shape)
print("y_window_train shape:", y_window_train.shape)
print("ts_window_train shape:", ts_window_train.shape)
print("X_window_test shape:", X_window_test.shape)
print("y_window_test shape:", y_window_test.shape)
print("ts_window_test shape:", ts_window_test.shape)

n_janelas, timesteps, n_features = X_window_train.shape

## 1D-CNN

In [ ]:
## class weights
classes = np.unique(y_window_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_window_train
)
class_weights = dict(zip(classes, class_weights))

## build 1D-CNN model
model = Sequential([
    Conv1D(filters=64, kernel_size=7, strides=1, activation='relu', padding="same", input_shape=(timesteps, n_features)),
    BatchNormalization(),
    Dropout(0.1),
    
    Conv1D(filters=128, kernel_size=5, strides=1, activation='relu', padding="same"),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    
    Conv1D(filters=256, kernel_size=3, strides=1, activation='relu', padding="same"),
    BatchNormalization(),
    Dropout(0.1),
    
    Conv1D(filters=128, kernel_size=3, strides=1, activation='relu', padding="same"),
    BatchNormalization(),
    Dropout(0.1),
    MaxPooling1D(pool_size=2),
    
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  # saída binária
])

optimizer = AdamW(
    learning_rate=1e-3,     # taxa inicial
    weight_decay=1e-4,      # regularização L2 desacoplada
    beta_1=0.9, beta_2=0.999, epsilon=1e-7
)

model.compile(
    optimizer=optimizer,
    loss='binary_focal_crossentropy',
    metrics=[
        'accuracy',
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall'),
        metrics.AUC(name='auc')
    ]
)

## callback for best model and earlystop
checkpoint = ModelCheckpoint(
    filepath="best_cnn_model.keras",
    monitor="val_auc",      # "val_auc"
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",      # critério de parada
    patience=20,              # quantas épocas sem melhora até parar
    mode="min",
    restore_best_weights=True,  # já carrega os melhores pesos no final
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

## train model
history = model.fit(
    X_window_train, y_window_train,
    validation_data=(X_window_test, y_window_test),
    # validation_split=0.1,
    # shuffle=False,
    epochs=100,
    batch_size=128,
    class_weight=class_weights,
    verbose=1,
    callbacks=[checkpoint, reduce_lr] #
)

## evaluate best model
best_model_cnn = load_model("best_cnn_model.keras")
loss, acc, precision, recall, auc = best_model_cnn.evaluate(X_window_test, y_window_test, verbose=0)
print(f"############ BEST MODEL METRICS ##############")
print(f"Acurácia no teste: {acc:.4f}")
print(f"Precisão   no teste: {precision:.4f}")
print(f"Recall     no teste: {recall:.4f}")
print(f"AUC        no teste: {auc:.4f}")

## evaluate last model
loss, acc, precision, recall, auc = model.evaluate(X_window_test, y_window_test, verbose=0)
print(f"############ LAST MODEL METRICS ##############")
print(f"Acurácia no teste: {acc:.4f}")
print(f"Precisão   no teste: {precision:.4f}")
print(f"Recall     no teste: {recall:.4f}")
print(f"AUC        no teste: {auc:.4f}")

In [ ]:
## Train metrics
def plot_metrics(history):
    metrics_list = ['loss', 'precision', 'recall', 'auc']
    plt.figure(figsize=(14, 8))

    for i, metric in enumerate(metrics_list, 1):
        plt.subplot(2, 2, i)
        plt.plot(history.history[metric], label=f'Treinamento {metric}')
        plt.plot(history.history[f'val_{metric}'], label=f'Validação {metric}')
        plt.title(metric.capitalize())
        plt.xlabel('Épocas')
        plt.ylabel(metric.capitalize())
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_metrics(history)


## ROC and Precision-Recall curves from BEST MODEL
y_pred_proba = best_model_cnn.predict(X_window_test).ravel()

# Curva ROC
fpr, tpr, _ = roc_curve(y_window_test, y_pred_proba)
roc_auc = calc_auc(fpr, tpr)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("Falso Positivo")
plt.ylabel("Verdadeiro Positivo")
plt.title("Curva ROC - Best Model")
plt.legend()
plt.grid(True)
plt.show()

## ROC and Precision-Recall curves from LAST MODEL
y_pred_proba = model.predict(X_window_test).ravel()

# Curva ROC
fpr, tpr, _ = roc_curve(y_window_test, y_pred_proba)
roc_auc = calc_auc(fpr, tpr)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("Falso Positivo")
plt.ylabel("Verdadeiro Positivo")
plt.title("Curva ROC - Last Model")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
## Best Model
y_pred_cnn = best_model_cnn.predict(X_window_test)

fig_pred_cnn_best = go.Figure()
fig_pred_cnn_best.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_pred_cnn.flatten(),
    name='Predict',
    mode="lines",
    line=dict(color='black')
))
fig_pred_cnn_best.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_window_test,
    name='Real',
    mode="lines",
    line=dict(color='red')
))
fig_pred_cnn_best.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Best Model CNN predicted vs Real Values"
)
fig_pred_cnn_best.show()

## Last Model
y_pred_cnn = model.predict(X_window_test)

fig_pred_cnn_last = go.Figure()
fig_pred_cnn_last.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_pred_cnn.flatten(),
    name='Predict',
    mode="lines",
    line=dict(color='black')
))
fig_pred_cnn_last.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_window_test,
    name='Real',
    mode="lines",
    line=dict(color='red')
))
fig_pred_cnn_last.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Last Model CNN predicted vs Real Values"
)
fig_pred_cnn_last.show()

## InceptionTime

In [ ]:
from tensorflow.keras import layers, models, metrics, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model

## class weights
classes = np.unique(y_window_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_window_train
)
class_weights = dict(zip(classes, class_weights))
print("Class weights:", class_weights)

## InceptionTime block
def inception_module(input_tensor, n_filters=32, kernel_sizes=(10, 20, 40), bottleneck_size=32):
    """
    Bloco Inception adaptado para séries temporais 1D
    """
    if bottleneck_size and input_tensor.shape[-1] > 1:
        x = layers.Conv1D(bottleneck_size, 1, padding='same', activation='relu')(input_tensor)
    else:
        x = input_tensor

    conv_list = []
    for k in kernel_sizes:
        conv_list.append(
            layers.Conv1D(filters=n_filters, kernel_size=k, padding='same', activation='relu')(x)
        )

    max_pool = layers.MaxPooling1D(pool_size=3, strides=1, padding='same')(input_tensor)
    conv_pool = layers.Conv1D(filters=n_filters, kernel_size=1, padding='same', activation='relu')(max_pool)
    conv_list.append(conv_pool)

    x = layers.Concatenate(axis=-1)(conv_list)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def build_inception_time(input_shape, n_classes=1):
    input_layer = Input(shape=input_shape)
    x = input_layer

    # Três blocos Inception empilhados
    for _ in range(3):
        x = inception_module(x, n_filters=32)
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.25)(x)
    output_layer = layers.Dense(n_classes, activation='sigmoid')(x)

    model = Model(inputs=input_layer, outputs=output_layer)
    return model

## build model
model = build_inception_time(input_shape=(timesteps, n_features), n_classes=1)

model.compile(
    optimizer='adam',
    loss='binary_focal_crossentropy',
    metrics=[
        'accuracy',
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall'),
        metrics.AUC(name='auc')
    ]
)
# model.summary()

## callbacks for best model and earlystop
checkpoint = ModelCheckpoint(
    filepath="best_inceptiontime_model.keras",
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

## train model
history = model.fit(
    X_window_train, y_window_train,
    validation_data=(X_window_test, y_window_test),
    epochs=100,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[checkpoint, reduce_lr],
    verbose=1
)

## evaluate best model
best_model_inceptiontime = load_model("best_inceptiontime_model.keras")
loss, acc, precision, recall, auc = best_model_inceptiontime.evaluate(X_window_test, y_window_test, verbose=0)

print(f"############ BEST MODEL METRICS ##############")
print(f"Acurácia:  {acc:.4f}")
print(f"Precisão:  {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"AUC:       {auc:.4f}")

## evaluate last model
loss, acc, precision, recall, auc = model.evaluate(X_window_test, y_window_test, verbose=0)
print(f"############ LAST MODEL METRICS ##############")
print(f"Acurácia no teste: {acc:.4f}")
print(f"Precisão   no teste: {precision:.4f}")
print(f"Recall     no teste: {recall:.4f}")
print(f"AUC        no teste: {auc:.4f}")

In [ ]:
## Train metrics
def plot_metrics(history):
    metrics_list = ['loss', 'precision', 'recall', 'auc']
    plt.figure(figsize=(14, 8))

    for i, metric in enumerate(metrics_list, 1):
        plt.subplot(2, 2, i)
        plt.plot(history.history[metric], label=f'Treinamento {metric}')
        plt.plot(history.history[f'val_{metric}'], label=f'Validação {metric}')
        plt.title(metric.capitalize())
        plt.xlabel('Épocas')
        plt.ylabel(metric.capitalize())
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_metrics(history)


## ROC and Precision-Recall curves from BEST MODEL
y_pred_proba = best_model_inceptiontime.predict(X_window_test).ravel()

# ROC
fpr, tpr, _ = roc_curve(y_window_test, y_pred_proba)
roc_auc = calc_auc(fpr, tpr)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("Falso Positivo")
plt.ylabel("Verdadeiro Positivo")
plt.title("Curva ROC - Best Model")
plt.legend()
plt.grid(True)
plt.show()

## ROC and Precision-Recall curves from LAST MODEL
y_pred_proba = model.predict(X_window_test).ravel()

# Curva ROC
fpr, tpr, _ = roc_curve(y_window_test, y_pred_proba)
roc_auc = calc_auc(fpr, tpr)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("Falso Positivo")
plt.ylabel("Verdadeiro Positivo")
plt.title("Curva ROC - Last Model")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
## Best Model
best_model_inceptiontime = load_model("best_inceptiontime_model.keras")
y_pred_inceptiontime = best_model_inceptiontime.predict(X_window_test)

fig_pred_inceptiontime_best = go.Figure()
fig_pred_inceptiontime_best.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_pred_inceptiontime.flatten(),
    name='Predict',
    mode="lines",
    line=dict(color='black')
))
fig_pred_inceptiontime_best.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_window_test,
    name='Real',
    mode="lines",
    line=dict(color='red')
))
fig_pred_inceptiontime_best.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Best Model InceptionTime predicted vs Real Values"
)
fig_pred_inceptiontime_best.show()

## Last Model
y_pred_inceptiontime = model.predict(X_window_test)

fig_pred_inceptiontime_last = go.Figure()
fig_pred_inceptiontime_last.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_pred_inceptiontime.flatten(),
    name='Predict',
    mode="lines",
    line=dict(color='black')
))
fig_pred_inceptiontime_last.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_window_test,
    name='Real',
    mode="lines",
    line=dict(color='red')
))
fig_pred_inceptiontime_last.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Last Model InceptionTime predicted vs Real Values"
)
fig_pred_inceptiontime_last.show()

## ResNet 1D

In [ ]:
## class weights
classes = np.unique(y_window_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_window_train
)
class_weights = dict(zip(classes, class_weights))
print("Pesos de classe:", class_weights)

## build ResNet
def build_resnet(input_shape):
    inputs = layers.Input(shape=input_shape)

    def residual_block(x, filters, kernel_size):
        shortcut = x

        x = layers.Conv1D(filters, kernel_size, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)

        x = layers.Conv1D(filters, kernel_size, padding="same")(x)
        x = layers.BatchNormalization()(x)

        # adjust shortcut dim
        if shortcut.shape[-1] != filters:
            shortcut = layers.Conv1D(filters, kernel_size=1, padding="same")(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)

        x = layers.Add()([x, shortcut])
        x = layers.Activation("relu")(x)
        return x

    x = residual_block(inputs, filters=64, kernel_size=8)
    x = residual_block(x, filters=128, kernel_size=5)
    x = residual_block(x, filters=128, kernel_size=3)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    return model

model = build_resnet((timesteps, n_features))
model.compile(
    optimizer="adam",
    loss="binary_focal_crossentropy",
    metrics=[
        "accuracy",
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.AUC(name="auc"),
    ]
)

## callbacks for model
checkpoint = ModelCheckpoint(
    filepath="best_resnet_model.keras",
    monitor="val_auc",
    save_best_only=True,
    mode="max",
    verbose=1
)
## define earlystop criteria
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=20,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

## training model
history = model.fit(
    X_window_train, y_window_train,
    validation_data=(X_window_test, y_window_test),
    epochs=100,
    batch_size=128,
    class_weight=class_weights,
    verbose=1,
    callbacks=[checkpoint, reduce_lr]
)

## evaluate best model
best_model_resnet = load_model("best_resnet_model.keras")
loss, acc, precision, recall, auc = best_model_resnet.evaluate(X_window_test, y_window_test, verbose=0)

print(f"############ BEST MODEL METRICS ##############")
print(f"Acurácia : {acc:.4f}")
print(f"Precisão : {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"AUC      : {auc:.4f}")

## evaluate last model
loss, acc, precision, recall, auc = model.evaluate(X_window_test, y_window_test, verbose=0)
print(f"############ LAST MODEL METRICS ##############")
print(f"Acurácia no teste: {acc:.4f}")
print(f"Precisão   no teste: {precision:.4f}")
print(f"Recall     no teste: {recall:.4f}")
print(f"AUC        no teste: {auc:.4f}")

In [ ]:
## Train metrics
def plot_metrics(history):
    metrics_list = ['loss', 'precision', 'recall', 'auc']
    plt.figure(figsize=(14, 8))

    for i, metric in enumerate(metrics_list, 1):
        plt.subplot(2, 2, i)
        plt.plot(history.history[metric], label=f'Treinamento {metric}')
        plt.plot(history.history[f'val_{metric}'], label=f'Validação {metric}')
        plt.title(metric.capitalize())
        plt.xlabel('Épocas')
        plt.ylabel(metric.capitalize())
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_metrics(history)


## ROC and Precision-Recall curves from BEST MODEL
y_pred_proba = best_model_resnet.predict(X_window_test).ravel()

# ROC
fpr, tpr, _ = roc_curve(y_window_test, y_pred_proba)
roc_auc = calc_auc(fpr, tpr)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("Falso Positivo")
plt.ylabel("Verdadeiro Positivo")
plt.title("Curva ROC")
plt.legend()
plt.grid(True)
plt.show()

## ROC and Precision-Recall curves from LAST MODEL
y_pred_proba = model.predict(X_window_test).ravel()

# Curva ROC
fpr, tpr, _ = roc_curve(y_window_test, y_pred_proba)
roc_auc = calc_auc(fpr, tpr)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("Falso Positivo")
plt.ylabel("Verdadeiro Positivo")
plt.title("Curva ROC - Last Model")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
## Best Model
best_model_resnet = load_model("best_resnet_model.keras")
y_pred_resnet = best_model_resnet.predict(X_window_test)

fig_pred_resnet_best = go.Figure()
fig_pred_resnet_best.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_pred_resnet.flatten(),
    name='Predict',
    mode="lines",
    line=dict(color='black')
))
fig_pred_resnet_best.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_window_test,
    name='Real',
    mode="lines",
    line=dict(color='red')
))
fig_pred_resnet_best.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Best Model ResNet predicted vs Real Values"
)
fig_pred_resnet_best.show()

## Last Model
y_pred_resnet = model.predict(X_window_test)

fig_pred_resent_last = go.Figure()
fig_pred_resent_last.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_pred_resnet.flatten(),
    name='Predict',
    mode="lines",
    line=dict(color='black')
))
fig_pred_resent_last.add_trace(go.Scatter(
    x=ts_window_test,
    y=y_window_test,
    name='Real',
    mode="lines",
    line=dict(color='red')
))
fig_pred_resent_last.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Last Model ResNet predicted vs Real Values"
)
fig_pred_resent_last.show()

## ResNet 2D

In [ ]:
# --- Ajuste de forma para Conv2D ---
X_window_train = np.expand_dims(X_window_train, axis=-1)  # (batch, time, features, 1)
X_window_test = np.expand_dims(X_window_test, axis=-1)

# --- Pesos de classe ---
classes = np.unique(y_window_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_window_train)
class_weights = dict(zip(classes, class_weights))
print("Pesos de classe:", class_weights)


# --- ResNet Conv2D ---
def build_resnet_conv2d(input_shape):
    inputs = layers.Input(shape=input_shape)

    def residual_block(x, filters):
        x_input = x

        # Conv1
        x = layers.Conv2D(filters, (7, 1), padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)

        # Conv2
        x = layers.Conv2D(filters, (5, 1), padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)

        # Conv3 + residual
        x = layers.Conv2D(filters, (3, 1), padding="same")(x)
        x = layers.BatchNormalization()(x)

        # Ajuste de dimensão se necessário
        if x_input.shape[-1] != x.shape[-1]:
            x_input = layers.Conv2D(filters, (1, 1), padding="same")(x_input)
            x_input = layers.BatchNormalization()(x_input)

        x = layers.Add()([x, x_input])
        x = layers.Activation("relu")(x)
        return x

    # Blocos residuais
    x = residual_block(inputs, 64)
    x = residual_block(x, 128)
    x = residual_block(x, 128)

    # Global Pooling + classificação
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    return model


# --- Construção e compilação ---
model = build_resnet_conv2d((timesteps, n_features, 1))
model.compile(
    optimizer="adam",
    loss="binary_focal_crossentropy",
    metrics=[
        "accuracy",
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.AUC(name="auc"),
    ]
)

# --- Callbacks ---
checkpoint = ModelCheckpoint(
    filepath="best_resnet2d_model.keras",
    monitor="val_auc",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=20,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

# --- Treinamento ---
history = model.fit(
    X_window_train, y_window_train,
    validation_data=(X_window_test, y_window_test),
    epochs=100,
    batch_size=128,
    class_weight=class_weights,
    verbose=1,
    callbacks=[checkpoint, reduce_lr]
)

# --- Avaliação do melhor modelo ---
best_model_resnet = load_model("best_resnet2d_model.keras")
loss, acc, precision, recall, auc = best_model_resnet.evaluate(X_window_test, y_window_test, verbose=0)

print(f"############ BEST MODEL METRICS ##############")
print(f"Acurácia : {acc:.4f}")
print(f"Precisão : {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"AUC      : {auc:.4f}")

# --- Avaliação do último modelo treinado ---
loss, acc, precision, recall, auc = model.evaluate(X_window_test, y_window_test, verbose=0)
print(f"############ LAST MODEL METRICS ##############")
print(f"Acurácia no teste: {acc:.4f}")
print(f"Precisão   no teste: {precision:.4f}")
print(f"Recall     no teste: {recall:.4f}")
print(f"AUC        no teste: {auc:.4f}")

# Apply SHAP

In [ ]:
window_size = 60
step = 30

start_date = pd.to_datetime("2025-06-16 12:00:00")
end_date = pd.to_datetime("2025-06-18 00:00:00")
mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_test_shap = df_dataset.loc[mask]

df_test_shap.sort_values(by=[timestamp],inplace=True)
print(df_test_shap.shape)
print("Falhas Test SHAP:",df_test_shap[df_test_shap["Falhas"]==1].shape[0])
print("Normal Test SHAP:",df_test_shap[df_test_shap["Falhas"]==0].shape[0])

X_test_shap = df_test_shap.drop(columns=["Timestamp", "Falhas"], axis=1)
y_test_shap = df_test["Falhas"]

# --- Normalização apenas das features ---
X_test_shap = pd.DataFrame(scaler.transform(X_test_shap), columns=X_test_shap.columns, index=X_test_shap.index)

# --- Reconstrução dos DataFrames completos (mantendo o Timestamp) ---
df_test_full_shap = pd.concat([df_test_shap["Timestamp"], X_test_shap, y_test_shap], axis=1)

# --- Criação das janelas ---
X_window_test_shap, y_window_test_shap, ts_window_test_shap = create_windows(df_test_full_shap, window_size, step)

n_janelas, timesteps, n_features = X_window_test_shap.shape

In [ ]:
import shap

# Calculate SHAP values
explainer = shap.DeepExplainer(best_model_resnet, X_window_train)
shap_values = explainer.shap_values(X_window_test_shap)
shap.summary_plot(shap_values, feature_names = X.columns)